In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json

import torch
import torchaudio
import torchaudio.transforms as T
from tqdm.auto import tqdm

In [3]:
ROOT_DIR = "/content/drive/MyDrive/GenAIAudioLDM"

In [4]:
with open(os.path.join(ROOT_DIR, "metadata.json"), "r", encoding="utf-8") as f:
    metadata = json.load(f)

In [5]:
from collections import Counter, defaultdict
import random

SEED = 42
TAG_LEVEL = 1
SPLIT_RATIOS = {"train": 0.7, "val": 0.1, "test": 0.2}
SAVE_SPLITS = True

def normalize_tags(tags):
    cleaned_tags = []
    seen = set()

    for tag in tags:
        cleaned_tag = str(tag).strip().lower()
        if cleaned_tag and cleaned_tag not in seen:
            cleaned_tags.append(cleaned_tag)
            seen.add(cleaned_tag)

    return cleaned_tags

def collect_samples_by_tag(metadata, tag_level=1):
    samples_by_tag = defaultdict(list)

    for audio_name, audio_meta in metadata.items():
        normalized_audio_meta = dict(audio_meta)
        normalized_audio_meta["tags"] = normalize_tags(audio_meta.get("tags", []))

        if len(normalized_audio_meta["tags"]) <= tag_level:
            continue

        split_tag = normalized_audio_meta["tags"][tag_level]
        samples_by_tag[split_tag].append((audio_name, normalized_audio_meta))

    return samples_by_tag

def compute_split_counts(total_samples, split_ratios):
    split_names = list(split_ratios)
    normalized_ratios = [split_ratios[name] for name in split_names]

    if any(ratio < 0 for ratio in normalized_ratios):
        raise ValueError("Split ratios cannot be negative.")

    ratio_sum = sum(normalized_ratios)
    if ratio_sum <= 0:
        raise ValueError("At least one split ratio must be positive.")

    normalized_ratios = [ratio / ratio_sum for ratio in normalized_ratios]
    raw_split_counts = {
        split_name: total_samples * ratio
        for split_name, ratio in zip(split_names, normalized_ratios)
    }
    split_counts = {
        split_name: int(raw_split_counts[split_name])
        for split_name in split_names
    }

    remaining = total_samples - sum(split_counts.values())
    while remaining > 0:
        split_name = max(
            split_names,
            key=lambda name: (
                raw_split_counts[name] - split_counts[name],
                -split_counts[name],
                split_ratios[name],
            ),
        )
        split_counts[split_name] += 1
        remaining -= 1

    return split_counts

def build_proportional_splits(
    metadata,
    tag_level=1,
    split_ratios=None,
    seed=42,
):
    if split_ratios is None:
        split_ratios = {"train": 0.7, "val": 0.1, "test": 0.2}

    samples_by_tag = collect_samples_by_tag(metadata, tag_level=tag_level)
    original_tag_counts = Counter({tag: len(samples) for tag, samples in samples_by_tag.items()})

    if not original_tag_counts:
        raise ValueError(f"No tags were found at TAG_LEVEL={tag_level}.")

    rng = random.Random(seed)
    splits = {split_name: {} for split_name in split_ratios}
    per_tag_split_counts = {}

    for tag, samples in sorted(samples_by_tag.items()):
        shuffled_samples = samples.copy()
        rng.shuffle(shuffled_samples)
        split_counts = compute_split_counts(len(shuffled_samples), split_ratios)
        per_tag_split_counts[tag] = split_counts

        offset = 0
        for split_name, split_size in split_counts.items():
            for audio_name, audio_meta in shuffled_samples[offset:offset + split_size]:
                splits[split_name][audio_name] = audio_meta
            offset += split_size

    split_tag_counts = {
        split_name: Counter(audio_meta["tags"][tag_level] for audio_meta in split_metadata.values())
        for split_name, split_metadata in splits.items()
    }
    split_sizes = {
        split_name: len(split_metadata)
        for split_name, split_metadata in splits.items()
    }
    total_files = sum(split_sizes.values())
    actual_split_ratios = {
        split_name: split_size / total_files
        for split_name, split_size in split_sizes.items()
    }

    return (
        splits,
        original_tag_counts,
        split_tag_counts,
        per_tag_split_counts,
        split_sizes,
        actual_split_ratios,
    )

(
    splits,
    original_tag_counts,
    split_tag_counts,
    per_tag_split_counts,
    split_sizes,
    actual_split_ratios,
) = build_proportional_splits(
    metadata,
    tag_level=TAG_LEVEL,
    split_ratios=SPLIT_RATIOS,
    seed=SEED,
)

train_metadata = splits["train"]
val_metadata = splits["val"]
test_metadata = splits["test"]

print(f"Built proportional train/val/test splits using TAG_LEVEL={TAG_LEVEL}.")
print("Requested split ratios:", SPLIT_RATIOS)
print("Overall split sizes:", split_sizes)
print(
    "Overall split ratios:",
    {split_name: round(ratio, 4) for split_name, ratio in actual_split_ratios.items()},
)
print("Original tag counts:", dict(sorted(original_tag_counts.items())))

if "allay" in per_tag_split_counts:
    print("Allay split counts:", per_tag_split_counts["allay"])

for split_name, split_metadata in splits.items():
    print(f"{split_name}: {len(split_metadata)} files")
    print(dict(sorted(split_tag_counts[split_name].items())))

if SAVE_SPLITS:
    for split_name, split_metadata in splits.items():
        split_metadata_path = os.path.join(
            ROOT_DIR,
            f"{split_name}_metadata_tag_level_{TAG_LEVEL}.json",
        )
        with open(split_metadata_path, "w", encoding="utf-8") as f:
            json.dump(split_metadata, f, indent=2, ensure_ascii=False)

        print(f"Saved {split_name} split to {split_metadata_path}")

{split_name: list(split_metadata.items())[:2] for split_name, split_metadata in splits.items()}

Built proportional train/val/test splits using TAG_LEVEL=1.
Requested split ratios: {'train': 0.7, 'val': 0.1, 'test': 0.2}
Overall split sizes: {'train': 1013, 'val': 141, 'test': 298}
Overall split ratios: {'train': 0.6977, 'val': 0.0971, 'test': 0.2052}
Original tag counts: {'allay': 21, 'amethyst': 31, 'anvil break': 1, 'anvil use': 1, 'armadillo': 6, 'armor': 2, 'axe': 2, 'axolotl': 4, 'azalea': 1, 'bamboo': 8, 'base': 1, 'bat': 1, 'beacon': 6, 'bee': 10, 'beehive': 5, 'bell': 3, 'blast': 1, 'blast far': 1, 'blastfurnace': 5, 'blaze': 5, 'blocks': 1, 'boat': 4, 'bogged': 1, 'bonemeal': 5, 'bowhit': 1, 'breath': 1, 'breeze': 23, 'brush': 9, 'bubble': 12, 'bucket': 11, 'calcite': 2, 'camel': 26, 'campfire': 6, 'cat': 17, 'cauldron': 3, 'cave': 23, 'chirp': 1, 'chiseled': 7, 'chorus': 6, 'conduit': 15, 'cow': 6, 'creator': 1, 'creator music box': 1, 'creeper': 1, 'crossbow': 7, 'deepslate': 3, 'dolphin': 20, 'drowned': 21, 'elytra': 1, 'enchantment': 3, 'end': 4, 'enderchest': 1, 'en

{'train': [('MOB_ALLAY_ITEM GIVEN1.ogg',
   {'tags': ['mob', 'allay', 'item given'],
    'prompt': 'generate me minecraft sound where mob allay item given',
    'path': '/content/drive/MyDrive/GenAIAudioLDM/preprocessed/mob_allay_item given.wav'}),
  ('MOB_ALLAY_IDLE WITHOUT ITEM4.ogg',
   {'tags': ['mob', 'allay', 'idle without item'],
    'prompt': 'generate me minecraft sound where mob allay idle without item',
    'path': '/content/drive/MyDrive/GenAIAudioLDM/preprocessed/mob_allay_idle without item.wav'})],
 'val': [('MOB_ALLAY_IDLE WITH ITEM2.ogg',
   {'tags': ['mob', 'allay', 'idle with item'],
    'prompt': 'generate me minecraft sound where mob allay idle with item',
    'path': '/content/drive/MyDrive/GenAIAudioLDM/preprocessed/mob_allay_idle with item.wav'}),
  ('MOB_ALLAY_ITEM GIVEN3.ogg',
   {'tags': ['mob', 'allay', 'item given'],
    'prompt': 'generate me minecraft sound where mob allay item given',
    'path': '/content/drive/MyDrive/GenAIAudioLDM/preprocessed/mob_alla

In [6]:
file_names = os.listdir(os.path.join(ROOT_DIR, "preprocessed"))
len(file_names)

556

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for file_name in tqdm(file_names):
    file_path = os.path.join(ROOT_DIR, "preprocessed", file_name)
    waveform, sample_rate = torchaudio.load(file_path)
    waveform = waveform.to(device)

    if waveform.shape[0] == 2:
        waveform = waveform.mean(dim=0)

    resampler = T.Resample(
        orig_freq=sample_rate,
        new_freq=16000
    ).to(device)

    with torch.inference_mode():
        resampled_waveform = resampler(waveform)
    
    torchaudio.save(os.path.join(ROOT_DIR, "preprocessed", file_name), resampled_waveform.cpu(), 16000)

  0%|          | 0/556 [00:00<?, ?it/s]

In [ ]:
# Install Hugging Face AudioLDM dependencies in Colab.
# AudioLDM is deprecated in recent diffusers releases. The last supported diffusers version is 0.33.1.
%pip install -q diffusers==0.33.1 transformers accelerate huggingface_hub safetensors

In [5]:
from pathlib import Path
from pprint import pprint

from diffusers import AudioLDMPipeline
from huggingface_hub import snapshot_download

AUDIO_LDM_REPO_ID = "cvssp/audioldm"  # Original AudioLDM, not AudioLDM 2.
HF_MODELS_DIR = os.path.join(ROOT_DIR, "huggingface_models")
AUDIO_LDM_DIR = os.path.join(HF_MODELS_DIR, AUDIO_LDM_REPO_ID.replace("/", "_"))

downloaded_model_dir = snapshot_download(
    repo_id=AUDIO_LDM_REPO_ID,
    local_dir=AUDIO_LDM_DIR,
    local_dir_use_symlinks=False,
)

print(f"Downloaded {AUDIO_LDM_REPO_ID} to: {downloaded_model_dir}")

def print_tree(root_dir, max_depth=2):
    root_dir = Path(root_dir)
    for path in sorted(root_dir.rglob("*")):
        depth = len(path.relative_to(root_dir).parts)
        if depth > max_depth:
            continue

        indent = "  " * (depth - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}{path.name}{suffix}")

print("\nDownloaded repo structure:")
print_tree(downloaded_model_dir, max_depth=2)

torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
pipe = AudioLDMPipeline.from_pretrained(
    downloaded_model_dir,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
)

def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())

def format_params(num_parameters):
    return f"{num_parameters / 1e6:.2f}M"

component_types = {
    component_name: type(component).__name__
    for component_name, component in pipe.components.items()
}

module_summary = {
    "text_encoder": {
        "class": type(pipe.text_encoder).__name__,
        "params": format_params(count_parameters(pipe.text_encoder)),
        "dtype": str(next(pipe.text_encoder.parameters()).dtype),
    },
    "unet": {
        "class": type(pipe.unet).__name__,
        "params": format_params(count_parameters(pipe.unet)),
        "dtype": str(next(pipe.unet.parameters()).dtype),
    },
    "vae": {
        "class": type(pipe.vae).__name__,
        "params": format_params(count_parameters(pipe.vae)),
        "dtype": str(next(pipe.vae.parameters()).dtype),
    },
    "vocoder": {
        "class": type(pipe.vocoder).__name__,
        "params": format_params(count_parameters(pipe.vocoder)),
        "dtype": str(next(pipe.vocoder.parameters()).dtype),
    },
}

unet_summary = {
    "sample_size": pipe.unet.config.sample_size,
    "in_channels": pipe.unet.config.in_channels,
    "out_channels": pipe.unet.config.out_channels,
    "layers_per_block": pipe.unet.config.layers_per_block,
    "block_out_channels": pipe.unet.config.block_out_channels,
    "down_block_types": pipe.unet.config.down_block_types,
    "up_block_types": pipe.unet.config.up_block_types,
    "cross_attention_dim": pipe.unet.config.cross_attention_dim,
}

vae_summary = {
    "in_channels": pipe.vae.config.in_channels,
    "out_channels": pipe.vae.config.out_channels,
    "latent_channels": pipe.vae.config.latent_channels,
    "block_out_channels": pipe.vae.config.block_out_channels,
    "down_block_types": pipe.vae.config.down_block_types,
    "up_block_types": pipe.vae.config.up_block_types,
}

scheduler_summary = {
    "class": type(pipe.scheduler).__name__,
    "num_train_timesteps": pipe.scheduler.config.num_train_timesteps,
    "beta_start": pipe.scheduler.config.beta_start,
    "beta_end": pipe.scheduler.config.beta_end,
    "prediction_type": pipe.scheduler.config.prediction_type,
}

print("\nPipeline class:", type(pipe).__name__)
print("Tokenizer class:", type(pipe.tokenizer).__name__)
print("\nPipeline components:")
pprint(component_types)

print("\nModule summary:")
pprint(module_summary)

print("\nUNet config summary:")
pprint(unet_summary)

print("\nVAE config summary:")
pprint(vae_summary)

print("\nScheduler summary:")
pprint(scheduler_summary)

print("\nUNet top-level blocks:")
for child_name, child_module in pipe.unet.named_children():
    print(f"- {child_name}: {type(child_module).__name__}")

print("\nText encoder top-level blocks:")
for child_name, child_module in pipe.text_encoder.named_children():
    print(f"- {child_name}: {type(child_module).__name__}")


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Downloaded cvssp/audioldm to: /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm

Downloaded repo structure:
.DS_Store
.cache/
  huggingface/
.gitattributes
README.md
model_index.json
scheduler/
  scheduler_config.json
text_encoder/
  config.json
  pytorch_model.bin
tokenizer/
  merges.txt
  special_tokens_map.json
  tokenizer.json
  tokenizer_config.json
  vocab.json
unet/
  config.json
  diffusion_pytorch_model.bin
vae/
  config.json
  diffusion_pytorch_model.bin
vocoder/
  config.json
  pytorch_model.bin


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

An error occurred while trying to fetch /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

An error occurred while trying to fetch /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
The AudioLDMPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 



Pipeline class: AudioLDMPipeline
Tokenizer class: RobertaTokenizerFast

Pipeline components:
{'scheduler': 'DDIMScheduler',
 'text_encoder': 'ClapTextModelWithProjection',
 'tokenizer': 'RobertaTokenizerFast',
 'unet': 'UNet2DConditionModel',
 'vae': 'AutoencoderKL',
 'vocoder': 'SpeechT5HifiGan'}

Module summary:
{'text_encoder': {'class': 'ClapTextModelWithProjection',
                  'dtype': 'torch.float16',
                  'params': '125.30M'},
 'unet': {'class': 'UNet2DConditionModel',
          'dtype': 'torch.float16',
          'params': '185.04M'},
 'vae': {'class': 'AutoencoderKL',
         'dtype': 'torch.float16',
         'params': '55.38M'},
 'vocoder': {'class': 'SpeechT5HifiGan',
             'dtype': 'torch.float16',
             'params': '55.26M'}}

UNet config summary:
{'block_out_channels': [128, 256, 384, 640],
 'cross_attention_dim': [128, 256, 384, 640],
 'down_block_types': ['DownBlock2D',
                      'CrossAttnDownBlock2D',
                    

In [13]:
from contextlib import nullcontext
from collections import defaultdict
import gc
import math
import numpy as np
import random

import torch.nn.functional as F
import torchaudio.functional as AF
from torch.cuda.amp import GradScaler
from torch.utils.data import DataLoader, Dataset
from transformers import SpeechT5FeatureExtractor

SEED = 42
TRAIN_METADATA_PATH = os.path.join(ROOT_DIR, "train_metadata_tag_level_1.json")
VAL_METADATA_PATH = os.path.join(ROOT_DIR, "val_metadata_tag_level_1.json")
TEST_METADATA_PATH = os.path.join(ROOT_DIR, "test_metadata_tag_level_1.json")

FINETUNE_OUTPUT_DIR = os.path.join(ROOT_DIR, "audioldm_finetune")
BEST_MODEL_DIR = os.path.join(FINETUNE_OUTPUT_DIR, "best_model")
HISTORY_PATH = os.path.join(FINETUNE_OUTPUT_DIR, "history.json")
TEST_METRICS_PATH = os.path.join(FINETUNE_OUTPUT_DIR, "test_metrics.json")

TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 3
NUM_WORKERS = 2
MAX_GRAD_NORM = 1.0

LEARNING_RATES = {
    "text_encoder": 5e-7,
    "unet": 1e-6,
    "vae": 5e-7,
    "vocoder": 5e-7,
}

LOSS_WEIGHTS = {
    "diffusion": 1.0,
    "vae_recon": 0.5,
    "vae_kl": 1e-7,
    "vocoder_teacher": 0.1,
    "waveform_recon": 0.1,
}
MEL_CLAMP_RANGE = (-12.0, 12.0)
LATENT_CLAMP_RANGE = (-30.0, 30.0)
USE_AMP = False  # Full AudioLDM fine-tuning is much more stable in fp32.

os.makedirs(FINETUNE_OUTPUT_DIR, exist_ok=True)

for metadata_path in [TRAIN_METADATA_PATH, VAL_METADATA_PATH, TEST_METADATA_PATH]:
    if not os.path.exists(metadata_path):
        raise FileNotFoundError(f"Missing split metadata file: {metadata_path}")

with open(TRAIN_METADATA_PATH, "r", encoding="utf-8") as f:
    train_metadata = json.load(f)

with open(VAL_METADATA_PATH, "r", encoding="utf-8") as f:
    val_metadata = json.load(f)

with open(TEST_METADATA_PATH, "r", encoding="utf-8") as f:
    test_metadata = json.load(f)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda" and USE_AMP
autocast_dtype = torch.float16 if use_amp else torch.float32

if "downloaded_model_dir" not in globals():
    downloaded_model_dir = snapshot_download(
        repo_id=AUDIO_LDM_REPO_ID,
        local_dir=AUDIO_LDM_DIR,
        local_dir_use_symlinks=False,
    )

if "pipe" in globals():
    del pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pipe = AudioLDMPipeline.from_pretrained(
    downloaded_model_dir,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)
pipe = pipe.to(device)
pipe.text_encoder.requires_grad_(True)
pipe.unet.requires_grad_(True)
pipe.vae.requires_grad_(True)
pipe.vocoder.requires_grad_(True)
pipe.text_encoder.train()

if use_amp:
    pipe.text_encoder.to(device, dtype=torch.float16)

if hasattr(pipe.unet, "enable_gradient_checkpointing"):
    pipe.unet.enable_gradient_checkpointing()

if hasattr(pipe.vae, "enable_gradient_checkpointing"):
    pipe.vae.enable_gradient_checkpointing()

if hasattr(pipe.text_encoder, "gradient_checkpointing_enable"):
    try:
        pipe.text_encoder.gradient_checkpointing_enable()
    except ValueError as error:
        print(f"Skipping text encoder gradient checkpointing: {error}")

vocoder_upsample_factor = np.prod(pipe.vocoder.config.upsample_rates) / pipe.vocoder.config.sampling_rate
AUDIO_LENGTH_IN_S = pipe.unet.config.sample_size * pipe.vae_scale_factor * vocoder_upsample_factor
TARGET_SAMPLE_RATE = pipe.vocoder.config.sampling_rate
TARGET_WAVEFORM_SAMPLES = int(AUDIO_LENGTH_IN_S * TARGET_SAMPLE_RATE)
TARGET_MEL_FRAMES = int(AUDIO_LENGTH_IN_S / vocoder_upsample_factor)

if TARGET_MEL_FRAMES % pipe.vae_scale_factor != 0:
    TARGET_MEL_FRAMES = int(np.ceil(TARGET_MEL_FRAMES / pipe.vae_scale_factor)) * pipe.vae_scale_factor

mel_feature_extractor = SpeechT5FeatureExtractor(
    sampling_rate=TARGET_SAMPLE_RATE,
    num_mel_bins=pipe.vocoder.config.model_in_dim,
)

def crop_or_pad_1d(tensor, target_length, random_crop=False):
    current_length = tensor.shape[-1]
    if current_length == target_length:
        return tensor

    if current_length < target_length:
        return F.pad(tensor, (0, target_length - current_length))

    start = 0
    if random_crop:
        start = random.randint(0, current_length - target_length)
    else:
        start = (current_length - target_length) // 2

    return tensor[start:start + target_length]

def crop_or_pad_2d(tensor, target_frames, random_crop=False):
    current_frames = tensor.shape[0]
    if current_frames == target_frames:
        return tensor

    if current_frames < target_frames:
        return F.pad(tensor, (0, 0, 0, target_frames - current_frames))

    start = 0
    if random_crop:
        start = random.randint(0, current_frames - target_frames)
    else:
        start = (current_frames - target_frames) // 2

    return tensor[start:start + target_frames]

def align_waveform_length(predicted_waveform, target_waveform):
    target_length = target_waveform.shape[-1]
    predicted_length = predicted_waveform.shape[-1]

    if predicted_length > target_length:
        return predicted_waveform[..., :target_length]

    if predicted_length < target_length:
        return F.pad(predicted_waveform, (0, target_length - predicted_length))

    return predicted_waveform

def sanitize_tensor(tensor, clamp_range=None):
    tensor = torch.nan_to_num(tensor.float(), nan=0.0, posinf=0.0, neginf=0.0)
    if clamp_range is not None:
        tensor = tensor.clamp(clamp_range[0], clamp_range[1])
    return tensor

class AudioLDMFineTuneDataset(Dataset):
    def __init__(self, metadata, split_name):
        self.items = list(metadata.items())
        self.split_name = split_name
        self.random_crop = split_name == "train"

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        audio_name, audio_meta = self.items[index]
        waveform, sample_rate = torchaudio.load(audio_meta["path"])

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0)
        else:
            waveform = waveform.squeeze(0)

        if sample_rate != TARGET_SAMPLE_RATE:
            waveform = AF.resample(waveform, sample_rate, TARGET_SAMPLE_RATE)

        waveform = crop_or_pad_1d(
            waveform.float(),
            TARGET_WAVEFORM_SAMPLES,
            random_crop=self.random_crop,
        )

        mel_features = mel_feature_extractor(
            audio_target=waveform.numpy(),
            sampling_rate=TARGET_SAMPLE_RATE,
            return_tensors="pt",
        )

        mel = mel_features.get("labels")
        if mel is None:
            mel = mel_features.get("input_values")
        if mel is None:
            raise KeyError("Could not find spectrogram output in SpeechT5FeatureExtractor result.")

        mel = crop_or_pad_2d(
            mel[0].float(),
            TARGET_MEL_FRAMES,
            random_crop=self.random_crop,
        )
        mel = sanitize_tensor(mel, clamp_range=MEL_CLAMP_RANGE)
        waveform = sanitize_tensor(waveform)

        return {
            "audio_name": audio_name,
            "prompt": audio_meta["prompt"],
            "waveform": waveform,
            "mel": mel,
        }

train_dataset = AudioLDMFineTuneDataset(train_metadata, split_name="train")
val_dataset = AudioLDMFineTuneDataset(val_metadata, split_name="val")
test_dataset = AudioLDMFineTuneDataset(test_metadata, split_name="test")

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
)

trainable_parameters = [
    {"params": pipe.text_encoder.parameters(), "lr": LEARNING_RATES["text_encoder"]},
    {"params": pipe.unet.parameters(), "lr": LEARNING_RATES["unet"]},
    {"params": pipe.vae.parameters(), "lr": LEARNING_RATES["vae"]},
    {"params": pipe.vocoder.parameters(), "lr": LEARNING_RATES["vocoder"]},
]

optimizer = torch.optim.AdamW(trainable_parameters, betas=(0.9, 0.999), weight_decay=1e-2)
updates_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, NUM_EPOCHS * updates_per_epoch),
)
scaler = GradScaler(enabled=use_amp)

def encode_prompts(prompts):
    text_inputs = pipe.tokenizer(
        prompts,
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )

    prompt_embeds = pipe.text_encoder(
        text_inputs.input_ids.to(device),
        attention_mask=text_inputs.attention_mask.to(device),
    ).text_embeds
    prompt_embeds = sanitize_tensor(F.normalize(prompt_embeds, dim=-1), clamp_range=(-10.0, 10.0))

    return prompt_embeds

def compute_batch_losses(batch):
    target_waveforms = sanitize_tensor(batch["waveform"].to(device))
    target_mels = sanitize_tensor(batch["mel"].to(device).unsqueeze(1), clamp_range=MEL_CLAMP_RANGE)
    prompt_embeds = encode_prompts(batch["prompt"])
    prediction_type = getattr(pipe.scheduler.config, "prediction_type", "epsilon") or "epsilon"
    autocast_context = (
        torch.autocast(device_type=device.type, dtype=autocast_dtype)
        if use_amp
        else nullcontext()
    )

    with autocast_context:
        posterior = pipe.vae.encode(target_mels)
        latents = sanitize_tensor(posterior.latent_dist.sample(), clamp_range=LATENT_CLAMP_RANGE)
        scaled_latents = sanitize_tensor(latents * pipe.vae.config.scaling_factor, clamp_range=LATENT_CLAMP_RANGE)

        noise = torch.randn_like(scaled_latents)
        timesteps = torch.randint(
            0,
            pipe.scheduler.config.num_train_timesteps,
            (scaled_latents.shape[0],),
            device=device,
            dtype=torch.long,
        )
        noisy_latents = sanitize_tensor(pipe.scheduler.add_noise(scaled_latents, noise, timesteps), clamp_range=LATENT_CLAMP_RANGE)
        model_pred = pipe.unet(
            noisy_latents,
            timesteps,
            encoder_hidden_states=None,
            class_labels=prompt_embeds,
        ).sample
        model_pred = sanitize_tensor(model_pred, clamp_range=LATENT_CLAMP_RANGE)

        if prediction_type == "epsilon":
            diffusion_target = noise
        elif prediction_type == "v_prediction":
            diffusion_target = pipe.scheduler.get_velocity(scaled_latents, noise, timesteps)
        else:
            raise ValueError(f"Unsupported scheduler prediction type: {prediction_type}")

        diffusion_target = sanitize_tensor(diffusion_target, clamp_range=LATENT_CLAMP_RANGE)
        diffusion_loss = F.mse_loss(model_pred.float(), diffusion_target.float())

        reconstructed_mels = sanitize_tensor(pipe.vae.decode(latents).sample, clamp_range=MEL_CLAMP_RANGE)
        vae_recon_loss = F.l1_loss(reconstructed_mels.float(), target_mels.float())
        vae_kl_loss = sanitize_tensor(posterior.latent_dist.kl()).mean()

        teacher_waveforms = sanitize_tensor(pipe.vocoder(target_mels.squeeze(1)))
        reconstructed_waveforms = sanitize_tensor(pipe.vocoder(reconstructed_mels.squeeze(1)))

        teacher_waveforms = align_waveform_length(teacher_waveforms, target_waveforms)
        reconstructed_waveforms = align_waveform_length(reconstructed_waveforms, target_waveforms)

        vocoder_teacher_loss = F.l1_loss(teacher_waveforms.float(), target_waveforms.float())
        waveform_recon_loss = F.l1_loss(reconstructed_waveforms.float(), target_waveforms.float())

        total_loss = (
            LOSS_WEIGHTS["diffusion"] * diffusion_loss
            + LOSS_WEIGHTS["vae_recon"] * vae_recon_loss
            + LOSS_WEIGHTS["vae_kl"] * vae_kl_loss
            + LOSS_WEIGHTS["vocoder_teacher"] * vocoder_teacher_loss
            + LOSS_WEIGHTS["waveform_recon"] * waveform_recon_loss
        )
        total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=1e4, neginf=1e4)

    loss_dict = {
        "total": total_loss,
        "diffusion": diffusion_loss,
        "vae_recon": vae_recon_loss,
        "vae_kl": vae_kl_loss,
        "vocoder_teacher": vocoder_teacher_loss,
        "waveform_recon": waveform_recon_loss,
    }
    non_finite_metrics = [name for name, value in loss_dict.items() if not torch.isfinite(value).all()]
    if non_finite_metrics:
        return None, {name: float(loss_dict[name].detach().float().mean()) for name in loss_dict}

    return loss_dict, None

def run_epoch(dataloader, training, split_name):
    metrics = defaultdict(float)
    num_examples = 0
    skipped_batches = 0

    pipe.unet.train(training)
    pipe.vae.train(training)
    pipe.vocoder.train(training)
    pipe.text_encoder.train(training)

    if training:
        optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(dataloader, desc=split_name)
    for step, batch in enumerate(progress_bar, start=1):
        with torch.set_grad_enabled(training):
            loss_dict, bad_loss_snapshot = compute_batch_losses(batch)

        if loss_dict is None:
            skipped_batches += 1
            progress_bar.set_postfix(skipped=skipped_batches)
            if training:
                optimizer.zero_grad(set_to_none=True)
            print(f"Skipped non-finite batch in {split_name}: {bad_loss_snapshot}")
            continue

        if training:
            scaled_loss = loss_dict["total"] / GRADIENT_ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()

            if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(dataloader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(pipe.text_encoder.parameters())
                    + list(pipe.unet.parameters())
                    + list(pipe.vae.parameters())
                    + list(pipe.vocoder.parameters()),
                    MAX_GRAD_NORM,
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                lr_scheduler.step()

        batch_size = batch["waveform"].shape[0]
        num_examples += batch_size
        for metric_name, metric_value in loss_dict.items():
            metrics[metric_name] += float(metric_value.detach()) * batch_size

        progress_bar.set_postfix(
            total=metrics["total"] / max(1, num_examples),
            diffusion=metrics["diffusion"] / max(1, num_examples),
        )

    epoch_metrics = {
        metric_name: metric_total / max(1, num_examples)
        for metric_name, metric_total in metrics.items()
    }
    epoch_metrics["skipped_batches"] = skipped_batches
    return epoch_metrics

print("Fine-tuning split sizes:", {
    "train": len(train_dataset),
    "val": len(val_dataset),
    "test": len(test_dataset),
})
print("Device:", device)
print("Audio length (s):", round(AUDIO_LENGTH_IN_S, 4))
print("Target waveform samples:", TARGET_WAVEFORM_SAMPLES)
print("Target mel frames:", TARGET_MEL_FRAMES)
print("Target mel bins:", pipe.vocoder.config.model_in_dim)
print("Learning rates:", LEARNING_RATES)
print("AMP enabled:", use_amp)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

An error occurred while trying to fetch /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

An error occurred while trying to fetch /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /content/drive/MyDrive/GenAIAudioLDM/huggingface_models/cvssp_audioldm/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
The AudioLDMPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


Skipping text encoder gradient checkpointing: ClapTextModelWithProjection does not support gradient checkpointing.
Fine-tuning split sizes: {'train': 1013, 'val': 141, 'test': 298}
Device: cuda
Audio length (s): 5.12
Target waveform samples: 81920
Target mel frames: 512
Target mel bins: 64
Learning rates: {'text_encoder': 5e-07, 'unet': 1e-06, 'vae': 5e-07, 'vocoder': 5e-07}
AMP enabled: False


/tmp/ipykernel_7698/3665562150.py:275: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)


In [14]:
history = []
best_val_loss = float("inf")
best_epoch = None

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    train_metrics = run_epoch(train_loader, training=True, split_name=f"train epoch {epoch}")
    val_metrics = run_epoch(val_loader, training=False, split_name=f"val epoch {epoch}")

    epoch_record = {
        "epoch": epoch,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "train": train_metrics,
        "val": val_metrics,
    }
    history.append(epoch_record)

    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    print("Train metrics:", train_metrics)
    print("Val metrics:", val_metrics)

    if val_metrics["total"] < best_val_loss:
        best_val_loss = val_metrics["total"]
        best_epoch = epoch
        pipe.save_pretrained(BEST_MODEL_DIR)
        print(f"Saved new best checkpoint to {BEST_MODEL_DIR}")

print(f"Best epoch: {best_epoch}")
print(f"Best val total loss: {best_val_loss:.6f}")
history[-1] if history else {}


Epoch 1/3


train epoch 1:   0%|          | 0/1013 [00:00<?, ?it/s]

val epoch 1:   0%|          | 0/141 [00:00<?, ?it/s]

Train metrics: {'total': 37.06392223920456, 'diffusion': 0.20352366680886963, 'vae_recon': 0.5484480620725016, 'vae_kl': 365607113.47666085, 'vocoder_teacher': 0.14674442306690486, 'waveform_recon': 0.10787776497221181, 'skipped_batches': 0}
Val metrics: {'total': 99.63533194965504, 'diffusion': 0.10940547460394562, 'vae_recon': 0.21923112890399093, 'vae_kl': 994049593.8353003, 'vocoder_teacher': 0.05346007137856585, 'waveform_recon': 0.06005622364633472, 'skipped_batches': 0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Train metrics: {'total': 21.359085619041416, 'diffusion': 0.07828906027574083, 'vae_recon': 0.14828962300189744, 'vae_kl': 211999775.6906466, 'vocoder_teacher': 0.03279556983594368, 'waveform_recon': 0.03395005022556, 'skipped_batches': 0}
Val metrics: {'total': 75.89094597753798, 'diffusion': 0.07577352777101985, 'vae_recon': 0.16304111803042973, 'vae_kl': 757271343.4767565, 'vocoder_teacher': 0.032015590111936394, 'waveform_recon': 0.03320543951658757, 'skipped_batches': 0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved new best checkpoint to /content/drive/MyDrive/GenAIAudioLDM/audioldm_finetune/best_model
Best epoch: 3
Best val total loss: 75.890946


{'epoch': 3,
 'learning_rate': 0.0,
 'train': {'total': 21.359085619041416,
  'diffusion': 0.07828906027574083,
  'vae_recon': 0.14828962300189744,
  'vae_kl': 211999775.6906466,
  'vocoder_teacher': 0.03279556983594368,
  'waveform_recon': 0.03395005022556,
  'skipped_batches': 0},
 'val': {'total': 75.89094597753798,
  'diffusion': 0.07577352777101985,
  'vae_recon': 0.16304111803042973,
  'vae_kl': 757271343.4767565,
  'vocoder_teacher': 0.032015590111936394,
  'waveform_recon': 0.03320543951658757,
  'skipped_batches': 0}}

In [15]:
if os.path.isdir(BEST_MODEL_DIR):
    del pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    pipe = AudioLDMPipeline.from_pretrained(
        BEST_MODEL_DIR,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )
    pipe = pipe.to(device)
    pipe.text_encoder.requires_grad_(True)
    pipe.unet.requires_grad_(True)
    pipe.vae.requires_grad_(True)
    pipe.vocoder.requires_grad_(True)
    pipe.text_encoder.eval()

    if use_amp:
        pipe.text_encoder.to(device, dtype=torch.float16)

test_metrics = run_epoch(test_loader, training=False, split_name="test")

with open(TEST_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved test metrics to", TEST_METRICS_PATH)
test_metrics

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The AudioLDMPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


test:   0%|          | 0/298 [00:00<?, ?it/s]

Saved test metrics to /content/drive/MyDrive/GenAIAudioLDM/audioldm_finetune/test_metrics.json


{'total': 35.98858354728194,
 'diffusion': 0.07201024171804084,
 'vae_recon': 0.15224164236811982,
 'vae_kl': 358336196.4633756,
 'vocoder_teacher': 0.03360431829597114,
 'waveform_recon': 0.03471392385533467,
 'skipped_batches': 0}

In [16]:
from IPython.display import Audio, display

INFERENCE_PROMPT = "minecraft zombie death"
NEGATIVE_PROMPT = "speech, vocals, distorted, clipping, low quality, noise burst"
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 3.0
NUM_WAVEFORMS_PER_PROMPT = 1
INFERENCE_AUDIO_LENGTH_IN_S = 5.12
INFERENCE_SEED = 42

INFERENCE_OUTPUT_DIR = os.path.join(ROOT_DIR, "audioldm_inference")
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

if "downloaded_model_dir" not in globals():
    downloaded_model_dir = snapshot_download(
        repo_id=AUDIO_LDM_REPO_ID,
        local_dir=AUDIO_LDM_DIR,
        local_dir_use_symlinks=False,
    )

inference_model_dir = BEST_MODEL_DIR if os.path.isdir(BEST_MODEL_DIR) else downloaded_model_dir
print(f"Using inference model: {inference_model_dir}")

if "inference_pipe" in globals():
    del inference_pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

inference_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
inference_pipe = AudioLDMPipeline.from_pretrained(
    inference_model_dir,
    torch_dtype=inference_dtype,
    low_cpu_mem_usage=True,
)
inference_pipe = inference_pipe.to(device)
inference_pipe.set_progress_bar_config(disable=False)

generator = torch.Generator(device=device.type).manual_seed(INFERENCE_SEED)

with torch.inference_mode():
    inference_output = inference_pipe(
        prompt=INFERENCE_PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        audio_length_in_s=INFERENCE_AUDIO_LENGTH_IN_S,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
        generator=generator,
        output_type="np",
    )

generated_audios = inference_output.audios
sample_rate = inference_pipe.vocoder.config.sampling_rate

saved_audio_paths = []
for audio_idx, generated_audio in enumerate(generated_audios):
    generated_audio = np.asarray(generated_audio, dtype=np.float32)
    generated_audio = np.clip(generated_audio, -1.0, 1.0)

    output_path = os.path.join(INFERENCE_OUTPUT_DIR, f"generated_{audio_idx:02d}.wav")
    torchaudio.save(output_path, torch.from_numpy(generated_audio).unsqueeze(0), sample_rate)
    saved_audio_paths.append(output_path)

    print(f"Generated audio {audio_idx}: {output_path}")
    display(Audio(generated_audio, rate=sample_rate))

saved_audio_paths

Using inference model: /content/drive/MyDrive/GenAIAudioLDM/audioldm_finetune/best_model


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The AudioLDMPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


  0%|          | 0/50 [00:00<?, ?it/s]

Generated audio 0: /content/drive/MyDrive/GenAIAudioLDM/audioldm_inference/generated_00.wav


['/content/drive/MyDrive/GenAIAudioLDM/audioldm_inference/generated_00.wav']